# CV2 Final Project - Script Runner (Colab GPU)

This notebook only executes the repository scripts in the same flow as the README:
1. Prepare data
2. Train
3. Evaluate
4. Inference demo

Training with GPU, if you want to use the scripts with cpu or locally check the README for instructions

## 0) Runtime Setup
In Colab, enable GPU first: Runtime > Change runtime type > T4 GPU (or similar).

In [1]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

# Auto-locate the repo in common Colab/Drive locations.
candidates = [
    Path('/content/CV2_Final_Project-1'),
    Path('/content/drive/MyDrive/CV2_Final_Project-1'),
    Path('/content/drive/MyDrive/Colab Notebooks/CV2_Final_Project-1'),
]

PROJECT_ROOT = next(
    (p for p in candidates if (p / 'requirements.txt').exists()),
    None,
 )

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not find project folder. Set PROJECT_ROOT manually to the folder containing requirements.txt'
    )

os.chdir(PROJECT_ROOT)
print('Working directory:', Path.cwd())

Mounted at /content/drive
Working directory: /content/drive/MyDrive/CV2_Final_Project-1


In [2]:
import torch

if torch.cuda.is_available():
    device_name = "cuda"
elif torch.backends.mps.is_available():
    device_name = "mps"
else:
    device_name = "cpu"
    
device = torch.device(device_name)
print(f"Code runs in {device}")

Code runs in cuda


In [3]:
import torch
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Torch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [4]:
# Install project dependencies from absolute path
!pip install -r "$PROJECT_ROOT/requirements.txt"

## 1) Prepare Data
Place `maps.zip` in the project root (or adjust the path below).

In [6]:
!python scripts/prepare_data.py --zip maps.zip --out data/raw

Dataset extracted to: data/raw
Created explicit splits in data/raw/maps: train=350, val=75, test=75


## 2) Train

In [ ]:
!python scripts/train.py --config configs/base.yaml

## 3) Evaluate

In [ ]:
!python scripts/evaluate.py --config configs/base.yaml --checkpoint checkpoints/best.pt

## 3b) Task 2 Improvement (Data Augmentation)

In [ ]:
!python scripts/train.py --config configs/improved_aug.yaml
!python scripts/evaluate.py --config configs/improved_aug.yaml --checkpoint checkpoints_aug/best.pt
!python - <<'PY'
import json
from pathlib import Path

base = json.loads(Path('outputs/metrics.json').read_text())
improved = json.loads(Path('outputs_aug/metrics.json').read_text())

print('Baseline   -> MAE: {mae:.6f} | PSNR: {psnr:.4f} | SSIM: {ssim:.4f}'.format(**base))
print('Improved   -> MAE: {mae:.6f} | PSNR: {psnr:.4f} | SSIM: {ssim:.4f}'.format(**improved))
PY

## 4) Inference Demo (Folder)

In [ ]:
!python scripts/infer.py --checkpoint checkpoints/best.pt --input data/raw/maps/test --output outputs/demo_test --paired-input

## Optional: Inference Demo (Single Image)

In [ ]:
from pathlib import Path
import subprocess

test_dir = Path('data/raw/maps/test')
candidates = sorted([
    p for p in test_dir.glob('*')
    if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
])
if not candidates:
    raise FileNotFoundError(f'No images found in {test_dir}')

first_img = candidates[0]
print('Using:', first_img)

subprocess.run([
    'python', 'scripts/infer.py',
    '--checkpoint', 'checkpoints/best.pt',
    '--input', str(first_img),
    '--output', 'outputs/demo_single.png',
    '--paired-input',
] , check=True)